In [ ]:
import torch
import pandas as pd

from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

/root/miniforge3/envs/MIMICIV/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Configuration
MODEL_NAME = 'Charangan/MedBERT'  # MedBERT model
# TODO: Insert hyperparameter like in the lab settings
BATCH_SIZE = 16
EPOCHS = 3
MAX_LEN = 512

# Load and preprocess dataset
landmark_df = pd.read_csv('your_landmark_dataset.csv')  # Replace with your dataset

# TODO: consider better
# Combine text columns
def combine_text(row):
    texts = []
    for col in ['med_text', 'diag_text', 'proc_text']:
        if pd.notna(row[col]):
            texts.append(row[col])
    return ' '.join(texts)

landmark_df['input_text'] = landmark_df.apply(combine_text, axis=1)

train_df, test_df = train_test_split(
    landmark_df[['input_text', 'death_in_90days']], 
    test_size=0.2, random_state=42, stratify=landmark_df['death_in_90days']
)

# Tokenization
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ClinicalDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding=True, max_length=max_length)
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ClinicalDataset(train_df['input_text'], train_df['death_in_90days'], tokenizer, MAX_LEN)
test_dataset = ClinicalDataset(test_df['input_text'], test_df['death_in_90days'], tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Model setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)

# Training loop
model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        inputs = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**inputs)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch + 1}/{EPOCHS}, Training Loss: {avg_loss:.4f}')

# Evaluation
model.eval()
predictions, true_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        inputs = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy()
        predictions.extend(probs)
        true_labels.extend(batch['labels'].cpu().numpy())

auc_score = roc_auc_score(true_labels, predictions)
print(f'MedBERT AUC: {auc_score:.3f}')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from scipy import interp
from sklearn.utils import resample

# Assume true_labels and predictions obtained from MedBERT evaluation

# Function to calculate confidence intervals with bootstrap
def bootstrap_auc_ci(y_true, y_pred, n_bootstraps=1000, alpha=0.95):
    bootstrapped_scores = []
    rng = np.random.RandomState(42)
    for _ in range(n_bootstraps):
        indices = rng.randint(0, len(y_pred), len(y_pred))
        if len(np.unique(y_true[indices])) < 2:
            continue
        score = auc(*roc_curve(y_true[indices], y_pred[indices])[:2])
        bootstrapped_scores.append(score)

    sorted_scores = np.array(bootstrapped_scores)
    sorted_scores.sort()
    lower = sorted_scores[int((1.0 - alpha) / 2 * len(sorted_scores))]
    upper = sorted_scores[int((alpha + (1.0 - alpha) / 2) * len(sorted_scores))]
    return lower, upper

# Compute ROC curve
fpr, tpr, thresholds = roc_curve(true_labels, predictions)
roc_auc = auc(fpr, tpr)
ci_lower, ci_upper = bootstrap_auc_ci(np.array(true_labels), np.array(predictions))

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.fill_between(fpr, tpr - (roc_auc - ci_lower), tpr + (ci_upper - roc_auc), alpha=0.2, color='blue', label=f'{95}% CI')

plt.plot([0, 1], [0, 1], color='grey', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('MedBERT ROC Curve with Confidence Interval')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()
